# Visualization — Lab 3 complexity audit

This notebook is the complexity analysis for the **visualization** module. Each nonempty source line appears below with an upper-bound or a note about its enclosing operation. The costs refer to the entire execution of that statement over enclosing loops; lines that form a multiline expression must be interpreted together. Detailed globally combined bounds are in `../../docs/complexity_summary.md`.

**Symbols.** `A` actors; `C=2` conditions; `R=C·A` recordings; `Z` ZIP entries; `N` samples; `K` 25 ms frame samples; `H` 10 ms hop samples; `W≈N/H` frames per recording; `W0` longer pYIN frames; `F=5` characteristics; `S=6` statistics; `d=4` classifier inputs; `M=2` models; `Pixels` number of saved image pixels; `T_pYIN` and `T_fit` are deliberately opaque library-level costs.

## Line-by-line analysis: `plots.py`

| Line | Source statement | Estimated cost / reason |
|---:|---|---|
| 1 | `"""Save distribution and ROC plots from paired neutral/sad recording summaries."""` | Comment/docstring — no runtime work |
| 3 | `from __future__ import annotations` | O(1) declaration/binding; function call cost discussed below |
| 5 | `from pathlib import Path` | O(1) declaration/binding; function call cost discussed below |
| 6 | `import matplotlib` | O(1) declaration/binding; function call cost discussed below |
| 7 | `matplotlib.use("Agg")` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 8 | `import matplotlib.pyplot as plt` | O(1) declaration/binding; function call cost discussed below |
| 9 | `import numpy as np` | O(1) declaration/binding; function call cost discussed below |
| 10 | `import pandas as pd` | O(1) declaration/binding; function call cost discussed below |
| 11 | `from sklearn.metrics import roc_curve, roc_auc_score` | O(1) declaration/binding; function call cost discussed below |
| 14 | `def plot_feature_comparison(neutral: pd.DataFrame, sad: pd.DataFrame,` | O(1) declaration/binding; function call cost discussed below |
| 15 | `output_dir: str \| Path, selected: list[tuple[str, str]]) -> list[Path]:` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 16 | `"""Save an independent paired-data plot for each feature/statistic selection."""` | Comment/docstring — no runtime work |
| 17 | `destination = Path(output_dir)` | O(m) path/string handling |
| 18 | `destination.mkdir(parents=True, exist_ok=True)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 19 | `paths = []` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 20 | `for characteristic, statistic in selected:` | O(F) iterations over characteristics |
| 21 | `x = neutral[(characteristic, statistic)].to_numpy(dtype=float)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 22 | `y = sad[(characteristic, statistic)].to_numpy(dtype=float)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 23 | `fig, ax = plt.subplots(figsize=(6, 4))` | O(1) figure setup (rendering later) |
| 24 | `for a, b in zip(x, y):` | O(A) paired actor lines |
| 25 | `if np.isfinite(a) and np.isfinite(b):` | O(1) condition/control unless evaluating an included collection |
| 26 | `ax.plot([0, 1], [a, b], color="0.75", alpha=0.55, linewidth=0.8)` | O(A) points per feature or O(R) ROC points |
| 27 | `ax.scatter(np.zeros(len(x)), x, s=28, color="#284B63", label="Neutral")` | O(A) points per feature or O(R) ROC points |
| 28 | `ax.scatter(np.ones(len(y)), y, s=28, color="#B05A3C", label="Sad")` | O(A) points per feature or O(R) ROC points |
| 29 | `ax.set_xticks([0, 1], ["Neutral", "Sad"])` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 30 | `ax.set_ylabel(f"{characteristic}: {statistic}")` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 31 | `ax.set_title("Paired actor comparison (acted emotions)")` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 32 | `ax.grid(axis="y", alpha=0.2)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 33 | `fig.tight_layout()` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 34 | `outfile = destination / f"paired_{characteristic.lower().replace(' ','_')}_{statistic.lower()}.png"` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 35 | `fig.savefig(outfile, dpi=160)` | O(Pixels) raster image encoding + rendering |
| 36 | `plt.close(fig)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 37 | `paths.append(outfile)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 38 | `return paths` | O(1) to return references (ignoring any nested expression) |
| 41 | `def plot_roc(predictions: pd.DataFrame, output_dir: str \| Path) -> Path:` | O(1) declaration/binding; function call cost discussed below |
| 42 | `"""Plot grouped cross-validated ROC curves (one curve per model)."""` | Comment/docstring — no runtime work |
| 43 | `destination = Path(output_dir)` | O(m) path/string handling |
| 44 | `destination.mkdir(parents=True, exist_ok=True)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 45 | `fig, ax = plt.subplots(figsize=(6.5, 5))` | O(1) figure setup (rendering later) |
| 46 | `for name, group in predictions.groupby("Model", sort=False):` | O(number of items) iteration; account for nested body cost |
| 47 | `fpr, tpr, _ = roc_curve(group["True"], group["P(Sad)"])` | O(R log R) sorting scores per model |
| 48 | `auc = roc_auc_score(group["True"], group["P(Sad)"])` | O(R log R) sorting scores per model |
| 49 | `ax.plot(fpr, tpr, linewidth=1.8, label=f"{name} (AUC {auc:.3f})")` | O(A) points per feature or O(R) ROC points |
| 50 | `ax.plot([0, 1], [0, 1], linestyle="--", color="0.5", label="Chance")` | O(A) points per feature or O(R) ROC points |
| 51 | `ax.set(xlabel="False positive rate", ylabel="True positive rate", title="Held-out-actor ROC curves")` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 52 | `ax.legend(fontsize=8)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 53 | `fig.tight_layout()` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 54 | `path = destination / "roc_grouped_cv.png"` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 55 | `fig.savefig(path, dpi=160)` | O(Pixels) raster image encoding + rendering |
| 56 | `plt.close(fig)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 57 | `return path` | O(1) to return references (ignoring any nested expression) |
| 60 | `def plot_benchmark_comparison(timing: pd.DataFrame, output_dir: str \| Path) -> Path:` | O(1) declaration/binding; function call cost discussed below |
| 61 | `"""Save a grouped chart of per-stage reference and optimized wall times."""` | Comment/docstring — no runtime work |
| 62 | `destination = Path(output_dir)` | O(m) path/string handling |
| 63 | `destination.mkdir(parents=True, exist_ok=True)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 64 | `fig, ax = plt.subplots(figsize=(9, 4.6))` | O(1) figure setup (rendering later) |
| 65 | `x = np.arange(len(timing))` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 66 | `width = 0.37` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 67 | `ax.bar(x - width / 2, timing["Reference [s]"], width, label="Reference")` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 68 | `ax.bar(x + width / 2, timing["Optimized [s]"], width, label="Optimized")` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 69 | `ax.set_xticks(x, timing["Stage"].str.replace(" (excluding identical pYIN)", "", regex=False), rotation=12, ha="ri...` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 70 | `ax.set_ylabel("Elapsed time (s), minimum of repeats")` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 71 | `ax.set_title("Independent benchmark stages — synthetic demo only")` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 72 | `ax.legend()` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 73 | `ax.grid(axis="y", alpha=0.2)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 74 | `fig.tight_layout()` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 75 | `out = destination / "benchmark_comparison.png"` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 76 | `fig.savefig(out, dpi=160)` | O(Pixels) raster image encoding + rendering |
| 77 | `plt.close(fig)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 78 | `return out` | O(1) to return references (ignoring any nested expression) |

## Global algorithm bounds

| Algorithm | End-to-end bound |
|---|---|
| `plot_feature_comparison` | O(F_selected·A) plotted measurements + O(F_selected·Pixels) image encoding. |
| `plot_roc` | O(M·R log R) for threshold sorting + O(Pixels) rendered output. |

**Optimization.** Only selected diagnostic figures are rendered, each one separately. Paths are returned and Matplotlib figures are closed after saving to prevent accumulation of figure objects.

In [ ]:
# Summary of the module: view the registered algorithm bounds without loading data.
complexities = [('plot_feature_comparison', 'O(F_selected·A) plotted measurements + O(F_selected·Pixels) image encoding.'), ('plot_roc', 'O(M·R log R) for threshold sorting + O(Pixels) rendered output.')]
for name, complexity in complexities:
    print(f"{name}: {complexity}")